# Feature correlation vs EOL (Week 3)

Load `cell_features.csv`, compute Pearson and Spearman correlations with **EOL**, rank features, and save a heatmap to `results/figures/feature_correlation.png`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError(f'Could not find data/raw/ starting from {here}')


ROOT = find_repo_root()
FEATURES_PATH = ROOT / 'data' / 'processed' / 'cell_features.csv'
FIGURE_PATH = ROOT / 'results' / 'figures' / 'feature_correlation.png'

LABEL_COLS = ('file_id', 'cell_id', 'EOL', 'initial_capacity')

print('Project root:', ROOT)
print('Input:', FEATURES_PATH)
print('Output:', FIGURE_PATH)

In [ ]:
cell_features = pd.read_csv(FEATURES_PATH)
feature_cols = [c for c in cell_features.columns if c not in LABEL_COLS]

assert len(cell_features) == 134
assert cell_features['file_id'].is_unique
assert cell_features[feature_cols].isna().sum().sum() == 0

print(f'Rows: {len(cell_features)}')
print(f'Feature columns: {len(feature_cols)}')

In [ ]:
def spearman(x: pd.Series, y: pd.Series) -> float:
    return x.rank().corr(y.rank(), method='pearson')


def correlations_with_eol(df: pd.DataFrame, features: list[str], target: str = 'EOL') -> pd.DataFrame:
    target_series = df[target]
    rows = []
    for col in features:
        series = df[col]
        rows.append(
            {
                'feature': col,
                'pearson': series.corr(target_series, method='pearson'),
                'spearman': spearman(series, target_series),
            }
        )
    out = pd.DataFrame(rows)
    out['abs_pearson'] = out['pearson'].abs()
    return out.sort_values('abs_pearson', ascending=False).reset_index(drop=True)


eol_corr = correlations_with_eol(cell_features, feature_cols)
print('Top 15 features by |Pearson| with EOL:')
display(eol_corr.head(15))

In [ ]:
import matplotlib

matplotlib.use('Agg')

corr_cols = feature_cols + ['EOL']
pearson_matrix = cell_features[corr_cols].corr(method='pearson')

n = len(corr_cols)
fig_h = max(10, n * 0.22)
fig, ax = plt.subplots(figsize=(fig_h, fig_h))

im = ax.imshow(pearson_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(corr_cols, rotation=90, ha='center', fontsize=7)
ax.set_yticklabels(corr_cols, fontsize=7)
ax.set_title('Pearson correlation — early-cycle features and EOL (n=134 cells)')

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Pearson r')

fig.tight_layout()
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=150)
plt.close(fig)

print(f'Wrote {FIGURE_PATH}')